# Crawl local + VAD trên HPC
Notebook gọi cùng CLI với terminal. Chỉ chạy các cell thuộc đúng máy hiện tại.

In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT = Path.cwd()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
CLI = str(PROJECT / 'run_pipeline.py')
print(PROJECT)

## Phần A - máy cá nhân có Internet
Crawl 320 giờ source để tạo buffer cho mục tiêu 200 giờ speech.

In [ ]:
subprocess.run([sys.executable, CLI, 'doctor', '--stage', 'crawl'], check=False)

In [ ]:
subprocess.run([sys.executable, CLI, 'crawl', '--target-source-hours', '320'], check=False)

In [ ]:
subprocess.run([sys.executable, CLI, 'export-crawl'], check=True)
print((PROJECT / 'transfer' / 'crawl_summary.json').read_text(encoding='utf-8'))

## Phần B - HPC
Chạy sau khi đã chuyển `data/raw` và `transfer` sang HPC.

In [ ]:
import os, getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF read token: ')
subprocess.run([sys.executable, CLI, 'doctor', '--stage', 'vad'], check=False)

In [ ]:
subprocess.run([sys.executable, CLI, 'import-crawl'], check=True)

In [ ]:
subprocess.run([sys.executable, CLI, 'vad', '--target-hours', '200'], check=False)

In [ ]:
subprocess.run([sys.executable, CLI, 'report'], check=True)
print((PROJECT / 'EDA_result' / 'REPORT.md').read_text(encoding='utf-8'))